# 04. Long Short-Term Memory (LSTM) Training

### Architecture:
- **Input**: Padded integer sequence.
- **Embedding**: Dense word embeddings.
- **Bidirectional LSTM**: Recurrent layers capturing both forward and backward contextual flow.
- **Dropout & Dense Layer**: Regularization and classification output.

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.preprocessing.nlp_pipeline import DatasetValidator, Vocabulary, LabelEncoder
from src.models.lstm import TextLSTM
from src.training.trainer import train_pytorch_model, TextDataset
from src.evaluation.metrics import evaluate_predictions

df = pd.read_csv('../data/sample_sentiment_benchmark.csv')
train_df, val_df, test_df = DatasetValidator.prepare_splits(df, 'text', 'sentiment')

vocab = Vocabulary()
vocab.build_vocab(train_df['clean_text'].tolist())

le = LabelEncoder()
le.fit(train_df['sentiment'].tolist())

X_train = vocab.batch_to_sequences(train_df['clean_text'].tolist(), max_len=64)
y_train = le.transform(train_df['sentiment'].tolist())
X_val = vocab.batch_to_sequences(val_df['clean_text'].tolist(), max_len=64)
y_val = le.transform(val_df['sentiment'].tolist())

train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=8, shuffle=True)
val_loader = DataLoader(TextDataset(X_val, y_val), batch_size=8)

# Initialize Bidirectional LSTM
model = TextLSTM(vocab_size=len(vocab), hidden_dim=64, num_classes=len(le), bidirectional=True)
print(model)

In [ ]:
# Train LSTM
train_results = train_pytorch_model(
    model, train_loader, val_loader, num_epochs=10, lr=1e-3
)
print(f"LSTM Training complete in {train_results['training_time_seconds']}s")

In [ ]:
# Test Set Evaluation
X_test = vocab.batch_to_sequences(test_df['clean_text'].tolist(), max_len=64)
y_test = le.transform(test_df['sentiment'].tolist())

with torch.no_grad():
    logits = model(torch.tensor(X_test, dtype=torch.long))
    preds = torch.argmax(logits, dim=1).numpy()

metrics = evaluate_predictions(y_test, preds, target_names=le.classes_)
print("LSTM Accuracy:", metrics["accuracy"])
print("LSTM Macro F1:", metrics["macro_f1"])